# Exercício: Modularização do Pipeline de ML

---

## Contexto

Você acabou de entrar em um time de ML. O cientista de dados anterior deixou um notebook com o pipeline completo de um modelo de clustering para regiões de entrega — tudo funcionando, mas **tudo misturado em um único bloco de código**.

Seu trabalho é **modularizar esse código**, separando cada responsabilidade em um módulo próprio, de forma que o projeto possa crescer, ser testado e ser mantido por outras pessoas.

Para isso, você vai usar o **Cookiecutter** — uma ferramenta que gera a estrutura de projeto a partir de um template padronizado.

---

## Objetivo

Ao final do exercício, você terá:

- Um projeto estruturado com módulos separados (`data`, `train`, `evaluate`, `predict`)
- Um pipeline orquestrado pelo `main.py`
- Um arquivo `outputs/metrics.json` com os resultados
- O projeto publicado no seu GitHub


---

## Ponto de Partida: O Código Monolítico

O código abaixo é o que você recebeu. Ele funciona — rode a célula, leia com atenção e entenda o que cada bloco faz. **Você não vai modificar esse código aqui.** O seu trabalho é recriá-lo de forma modularizada no projeto gerado pelo Cookiecutter.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import geopy.distance
import json
import os

# ---- Gerar dados sintéticos ----
np.random.seed(42)
n = 300
lats = np.random.normal(-23.55, 0.15, n)
lngs = np.random.normal(-46.63, 0.15, n)
points = np.column_stack([lats, lngs])

# ---- Preparar features ----
scaler = StandardScaler()
features = scaler.fit_transform(points)

# ---- Treinar modelo ----
n_clusters = 5
model = KMeans(n_clusters=n_clusters, random_state=42)
model.fit(features)

# ---- Avaliar ----
inertia = model.inertia_
sil = silhouette_score(features, model.labels_)
metrics = {
    "inertia": round(inertia, 4),
    "silhouette_score": round(sil, 4),
    "n_clusters": n_clusters
}

# ---- Predizer ----
lat_test, lng_test = -23.55, -46.63
point_scaled = scaler.transform([[lat_test, lng_test]])
cluster = int(model.predict(point_scaled)[0])
centroid_scaled = model.cluster_centers_[cluster]
centroid_original = scaler.inverse_transform([centroid_scaled])[0]
distance_km = geopy.distance.geodesic(
    (lat_test, lng_test),
    (centroid_original[0], centroid_original[1])
).km
prediction = {
    "cluster": cluster,
    "distance_km": round(distance_km, 4)
}

# ---- Resultado ----
output = {"metrics": metrics, "sample_prediction": prediction}
print(json.dumps(output, indent=2))

---

## O Exercício

Você vai recriar esse pipeline de forma modularizada, usando o Cookiecutter para gerar a estrutura do projeto.

O código acima deve ser dividido em **4 módulos**:

| Módulo | Arquivo | Responsabilidade |
|--------|---------|------------------|
| Dados | `src/data.py` | Carregar o CSV e normalizar os features |
| Treino | `src/train.py` | Treinar o modelo KMeans |
| Avaliação | `src/evaluate.py` | Calcular métricas do modelo |
| Predição | `src/predict.py` | Classificar um ponto e calcular distância |

O `main.py` (já gerado pelo template) conecta todos os módulos e salva o resultado em `outputs/metrics.json`.

---

### Passo 1 — Instalar o Cookiecutter


In [ ]:
!pip install cookiecutter -q

### Passo 2 — Gerar o projeto

No terminal (fora do Jupyter), na pasta onde você quer criar o projeto, execute:

```bash
cookiecutter gh:weslleymoura/mlops-template
```

O Cookiecutter vai perguntar:
- `project_name`: deixe `mlops-pipeline` (padrão)
- `github_username`: coloque o seu usuário do GitHub
- `author_name`: coloque o seu nome

Isso vai gerar a pasta `mlops-pipeline/` com toda a estrutura pronta.

---

### Passo 3 — Instalar dependências e gerar os dados

```bash
cd mlops-pipeline
pip install -r requirements.txt
python generate_data.py
```

---

### Passo 4 — Implementar os módulos

Abra a pasta `mlops-pipeline/` no VS Code e implemente as funções em cada arquivo de `src/`. Cada função tem uma docstring explicando o que deve fazer e o que deve retornar.

**`src/data.py`**
- `load_data(filepath)` → lê o CSV e retorna `np.ndarray` de shape `(n, 2)`
- `prepare_features(points)` → normaliza com `StandardScaler` e retorna `(features, scaler)`

**`src/train.py`**
- `train_model(features, n_clusters, random_state)` → treina e retorna o modelo `KMeans`

**`src/evaluate.py`**
- `evaluate_model(model, features)` → retorna `dict` com `inertia`, `silhouette_score`, `n_clusters`

**`src/predict.py`**
- `predict(model, scaler, lat, lng)` → retorna `dict` com `cluster` e `distance_km`

---

### Passo 5 — Executar o pipeline

```bash
python main.py
```

Se tudo estiver correto, você verá o resultado no terminal e o arquivo `outputs/metrics.json` será gerado.

---

### Passo 6 — Subir no GitHub

Crie um repositório público no GitHub chamado **`mlops-pipeline`** e suba o projeto:

```bash
git init
git add .
git commit -m "exercicio de modularizacao"
git remote add origin https://github.com/SEU-USERNAME/mlops-pipeline.git
git push -u origin main
```

> O repositório precisa ser **público**.

---

### Entrega

Informe ao professor o seu **usuário do GitHub**. A correção será feita automaticamente lendo o arquivo:

```
https://raw.githubusercontent.com/SEU-USERNAME/mlops-pipeline/main/outputs/metrics.json
```
